In [1]:
import os
import re
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Force bypass of TensorFlow to avoid previous environment errors
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

/home/jovyan/vault/AI_CEO_NLP/ai_ceo_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_864/1376416935.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [4]:
# 1. Pipeline Initialization
def setup_intelligence_engine():
    print("Loading Embedding Model")
    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-base-en-v1.5",
        model_kwargs={'device': 'cuda'},
        encode_kwargs={'normalize_embeddings': True}
    )

    print("Connecting to ChromaDB")
    vectorstore = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

    print("Loading Llama 3.1 8B")
    
    # Model Path
    model_id = "/home/jovyan/models/meta-llama/Llama-3.1-8B-Instruct"
    
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )

    # Temperature is set to 0.1 to prevent hallucinations and force factual reasoning
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=1500,
        temperature=0.1,
        return_full_text=False
    )
    llm = HuggingFacePipeline(pipeline=pipe)
    
    return retriever, llm

# 2. Prompt Engineering for JSON Output
# We demand JSON so the Streamlit dashboard can parse it easily later.

system_prompt = """You are an AI Strategic Intelligence Agent acting as the CEO advisor for NVIDIA.
Use ONLY the provided context to answer the prompt. If the context does not contain the answer, say so. Do not use outside knowledge.

Context:
{context}

Generate a strategic intelligence report formatted STRICTLY as a valid JSON object. Do not include markdown blocks, preamble, or conversational text. The JSON must have the following structure:
{{
  "opportunities": [
    {{"title": "...", "impact_level": "High/Medium/Low", "evidence": "...", "confidence_score": 1_to_100}}
  ],
  "risks": [
    {{"title": "...", "category": "...", "severity_level": "High/Medium/Low", "evidence": "...", "confidence_score": 1_to_100}}
  ],
  "trends": [
    {{"title": "...", "description": "..."}}
  ],
  "recommendations": [
        {{
          "recommendation": "...",
          "priority": "High/Medium/Low",
          "supporting_evidence": ["Extract explicit sentence from context here", "Extract another explicit sentence here"],
          "expected_impact": ["Predict specific business impact here", "Predict another business impact here"],
          "risk_assessment": {{"financial_risk": "...", "operational_risk": "...", "strategic_risk": "..."}}
        }}
      ],
  "ceo_briefing": {{
    "what_happened": "...",
    "why_it_matters": "...",
    "next_steps": "..."
  }}
}}
"""

# 3. Main Execution Block

if __name__ == "__main__":
    retriever, llm = setup_intelligence_engine()
    prompt = PromptTemplate.from_template(system_prompt)
    
    def format_docs(docs):
        formatted = []
        for d in docs:
            source = d.metadata.get('source', 'Unknown')
            url = d.metadata.get('url', 'No URL')
            formatted.append(f"Source: {source} ({url})\nContent: {d.page_content}\n")
        return "\n".join(formatted)

    rag_chain = (
        {"context": retriever | format_docs}
        | prompt
        | llm
        | StrOutputParser()
    )

    print("\nAnalyzing repository and generating strategic JSON report. This may take a minute")
    raw_output = rag_chain.invoke("What are the most critical strategic developments, risks, and opportunities for NVIDIA right now?")
    
    output_file = "ceo_intelligence_report.json"
    
    try:
        # 1. Find where the first JSON object starts
        start_idx = raw_output.find('{')
        if start_idx == -1:
            raise ValueError("No '{' found in the output.")
            
        # 2. Slice the string to start exactly at the first '{'
        json_string = raw_output[start_idx:]
        
        # 3. raw_decode parses the first complete object and ignores everything after it
        decoder = json.JSONDecoder()
        parsed_json, _ = decoder.raw_decode(json_string)
        
        # 4. Save the clean JSON
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(parsed_json, f, indent=4)
            
        print(f"\nSuccess. Strategic analysis complete. Data saved to {output_file}.")
        
    except (json.JSONDecodeError, ValueError) as e:
        print(f"\nError parsing JSON: {e}\nSee raw output below:\n")
        print(raw_output)
        with open("ceo_intelligence_report_RAW.txt", "w", encoding="utf-8") as f:
            f.write(raw_output)

Loading Embedding Model


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7996.73it/s]


Connecting to ChromaDB
Loading Llama 3.1 8B


Loading weights: 100%|██████████| 291/291 [00:01<00:00, 163.58it/s]



Analyzing repository and generating strategic JSON report. This may take a minute

Success. Strategic analysis complete. Data saved to ceo_intelligence_report.json.
